In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# 1. LOAD DATA
print("="*60)
print("STEP 1: LOADING DATA")
print("="*60)

df = pd.read_csv('../data/preprocessed/city_day_with_weather.csv')

print(f"\n Dataset loaded successfully!")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")

# 2. BASIC DATASET INFO
print("\n" + "="*60)
print("STEP 2: BASIC INFORMATION")
print("="*60)
print("\nFirst 5 rows:")
print(df.head())
print("\nLast 5 rows:")
print(df.tail())
print("\nData types:")
print(df.dtypes)
print("\nDataset memory usage:")
print(df.memory_usage(deep=True).sum() / 1024**2, "MB")

# 3. MISSING VALUES ANALYSIS
print("\n" + "="*60)
print("STEP 3: MISSING VALUES ANALYSIS")
print("="*60)

missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': missing.values,
    'Missing_Percentage': missing_percent.values
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Percentage', ascending=False)
print("\nMissing values summary:")
print(missing_df)
print(f"\nTotal missing values: {df.isnull().sum().sum()}")
print(f"Percentage of dataset that is missing: {(df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100:.2f}%")

# 4. DUPLICATE ROWS
print("\n" + "="*60)
print("STEP 4: DUPLICATE ROWS CHECK")
print("="*60)

duplicates = df.duplicated().sum()
print(f"\nTotal duplicate rows: {duplicates}")
if duplicates > 0:
    print("Duplicate rows found!")
    print(df[df.duplicated(keep=False)].head())
else:
    print("No duplicate rows found!")

# 5. STATISTICAL SUMMARY
print("\n" + "="*60)
print("STEP 5: STATISTICAL SUMMARY")
print("="*60)
print("\nNumeric columns statistics:")
print(df.describe())

# 6. UNIQUE VALUES
print("\n" + "="*60)
print("STEP 6: UNIQUE VALUES CHECK")
print("="*60)
print(f"\nUnique Cities: {df['City'].nunique()}")
print("Cities in dataset:")
print(df['City'].value_counts())
print(f"\nAQI Buckets: {df['AQI_Bucket'].nunique()}")
print("AQI Distribution:")
print(df['AQI_Bucket'].value_counts())

# 7. OUTLIERS CHECK (using IQR method)
print("\n" + "="*60)
print("STEP 7: OUTLIERS DETECTION")
print("="*60)

def count_outliers(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return len(outliers)
print("\nOutliers in numeric columns (using IQR method):")
numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    outlier_count = count_outliers(col)
    outlier_percent = (outlier_count / len(df)) * 100
    if outlier_count > 0:
        print(f"  {col}: {outlier_count} outliers ({outlier_percent:.2f}%)")

# 8. DATA RANGES
print("\n" + "="*60)
print("STEP 8: DATA RANGES")
print("="*60)
print("\nMin and Max values for key columns:")
key_columns = ['PM2.5', 'PM10', 'NO2', 'O3', 'CO', 'SO2', 'AQI', 'Temperature', 'Humidity', 'Wind_Speed']
for col in key_columns:
    if col in df.columns:
        print(f"{col:15} → Min: {df[col].min():10.2f}, Max: {df[col].max():10.2f}, Mean: {df[col].mean():10.2f}")

# 9. CORRELATION ANALYSIS
print("\n" + "="*60)
print("STEP 9: CORRELATION WITH TARGET (AQI)")
print("="*60)

# Check correlation with AQI
numeric_df = df.select_dtypes(include=[np.number])
correlations = numeric_df.corr()['AQI'].sort_values(ascending=False)
print("\nTop 10 features correlated with AQI:")
print(correlations.head(10))

# 10. SUMMARY & RECOMMENDATIONS
print("\n" + "="*60)
print("STEP 10: EDA SUMMARY & RECOMMENDATIONS")
print("="*60)
print("\n DATASET QUALITY REPORT:")
print(f"  • Rows: {df.shape[0]:,}")
print(f"  • Columns: {df.shape[1]}")
print(f"  • Missing values: {df.isnull().sum().sum():,} cells")
print(f"  • Duplicates: {duplicates}")
print(f"  • Unique cities: {df['City'].nunique()}")


print("\n Ready for data cleaning!")

STEP 1: LOADING DATA

 Dataset loaded successfully!
Shape: 29531 rows × 20 columns

STEP 2: BASIC INFORMATION

First 5 rows:
        City        Date  PM2.5  PM10     NO    NO2    NOx  NH3     CO    SO2  \
0  Ahmedabad  2015-01-01    NaN   NaN   0.92  18.22  17.15  NaN   0.92  27.64   
1  Ahmedabad  2015-01-02    NaN   NaN   0.97  15.69  16.46  NaN   0.97  24.55   
2  Ahmedabad  2015-01-03    NaN   NaN  17.40  19.30  29.70  NaN  17.40  29.07   
3  Ahmedabad  2015-01-04    NaN   NaN   1.70  18.48  17.97  NaN   1.70  18.59   
4  Ahmedabad  2015-01-05    NaN   NaN  22.10  21.42  37.76  NaN  22.10  39.33   

       O3  Benzene  Toluene  Xylene  AQI AQI_Bucket  Month  Temperature  \
0  133.36     0.00     0.02    0.00  NaN        NaN      1         16.6   
1   34.06     3.68     5.50    3.77  NaN        NaN      1         23.1   
2   30.70     6.80    16.40    2.25  NaN        NaN      1         15.7   
3   36.08     4.43    10.14    1.00  NaN        NaN      1         16.5   
4   39.31    

In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# LOAD DATA
print("="*60)
print("LOADING DATA FOR CLEANING")
print("="*60)
df = pd.read_csv('../data/preprocessed/city_day_with_weather.csv')
print(f"\nOriginal shape: {df.shape}")

# STEP 1: ANALYZE MISSING VALUES BY COLUMN
print("\n" + "="*60)
print("STEP 1: MISSING VALUES BY COLUMN")
print("="*60)
missing_analysis = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum().values,
    'Missing_Percentage': (df.isnull().sum().values / len(df) * 100).round(2)
})
missing_analysis = missing_analysis[missing_analysis['Missing_Count'] > 0].sort_values(
    'Missing_Percentage', ascending=False
)
print("\nMissing values by column:")
print(missing_analysis.to_string())

# STEP 2: DECIDE WHICH COLUMNS TO KEEP
print("\n" + "="*60)
print("STEP 2: COLUMN SELECTION")
print("="*60)

# Columns with >50% missing should be dropped
threshold = 50
columns_to_drop = missing_analysis[missing_analysis['Missing_Percentage'] > threshold]['Column'].tolist()
print(f"\nColumns with >{threshold}% missing (will be dropped):")
if columns_to_drop:
    for col in columns_to_drop:
        missing_pct = missing_analysis[missing_analysis['Column'] == col]['Missing_Percentage'].values[0]
        print(f"  • {col}: {missing_pct}% missing")
else:
    print("  None found - Good!")

# Columns to keep
important_cols = ['City', 'Date', 'AQI', 'AQI_Bucket', 'Month', 'Temperature', 'Humidity', 'Wind_Speed']
pollutant_cols = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']
print(f"\nColumns to keep:")
print(f"  • Important: {important_cols}")
print(f"  • Pollutants: {pollutant_cols}")

# STEP 3: DROP ROWS WHERE AQI IS NaN (TARGET)
print("\n" + "="*60)
print("STEP 3: DROP ROWS WITH MISSING TARGET (AQI)")
print("="*60)
initial_rows = len(df)
df = df.dropna(subset=['AQI'])
rows_removed = initial_rows - len(df)
print(f"\nRows with missing AQI: {rows_removed}")
print(f"Rows remaining: {len(df)}")

# STEP 4: FILL MISSING VALUES FOR POLLUTANTS
print("\n" + "="*60)
print("STEP 4: FILL MISSING POLLUTANT VALUES")
print("="*60)
# For each city, fill missing values with median of that city's data
print("\nFilling missing pollutant values with city-wise median...")
for pollutant in pollutant_cols:
    if pollutant in df.columns:
        # Fill with city median first
        df[pollutant] = df.groupby('City')[pollutant].transform(
            lambda x: x.fillna(x.median())
        )
        # If still NaN (city has no data), fill with overall median
        df[pollutant] = df[pollutant].fillna(df[pollutant].median())
        
        missing_after = df[pollutant].isnull().sum()
        print(f"  {pollutant}: {missing_after} NaN remaining")

# STEP 5: FILL MISSING VALUES FOR WEATHER
print("\n" + "="*60)
print("STEP 5: FILL MISSING WEATHER VALUES")
print("="*60)
weather_cols = ['Temperature', 'Humidity', 'Wind_Speed']
for weather in weather_cols:
    if weather in df.columns:
        # Fill with city median (weather is location-dependent)
        df[weather] = df.groupby('City')[weather].transform(
            lambda x: x.fillna(x.median())
        )
        # If still NaN, fill with overall median
        df[weather] = df[weather].fillna(df[weather].median())
        
        missing_after = df[weather].isnull().sum()
        print(f"  {weather}: {missing_after} NaN remaining")

# STEP 6: VERIFY ALL MISSING VALUES HANDLED
print("\n" + "="*60)
print("STEP 6: VERIFICATION")
print("="*60)

total_missing = df.isnull().sum().sum()
print(f"\nTotal missing values after cleaning: {total_missing}")

if total_missing == 0:
    print(" ALL MISSING VALUES HANDLED!")
else:
    print(f" Warning: {total_missing} missing values still exist")
    print(df.isnull().sum())

# STEP 7: DATA QUALITY REPORT
print("\n" + "="*60)
print("STEP 7: CLEANED DATA QUALITY REPORT")
print("="*60)

print(f"\n Shape: {df.shape}")
print(f" Rows: {len(df):,}")
print(f" Columns: {len(df.columns)}")
print(f" Missing values: {total_missing}")
print(f" Duplicates: {df.duplicated().sum()}")
print(f" Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nData types:")
print(df.dtypes)


# Save the intermediate cleaned dataset
df.to_csv('../data/preprocessed/city_day_cleaned_v1.csv', index=False)
print("\n Saved: data/preprocessed/city_day_cleaned_v1.csv")

LOADING DATA FOR CLEANING

Original shape: (29531, 20)

STEP 1: MISSING VALUES BY COLUMN

Missing values by column:
        Column  Missing_Count  Missing_Percentage
13      Xylene          18109               61.32
3         PM10          11140               37.72
7          NH3          10328               34.97
12     Toluene           8041               27.23
11     Benzene           5623               19.04
14         AQI           4681               15.85
15  AQI_Bucket           4681               15.85
2        PM2.5           4598               15.57
6          NOx           4185               14.17
10          O3           4022               13.62
9          SO2           3854               13.05
5          NO2           3585               12.14
4           NO           3582               12.13
8           CO           2059                6.97

STEP 2: COLUMN SELECTION

Columns with >50% missing (will be dropped):
  • Xylene: 61.32% missing

Columns to keep:
  • Important: ['

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns

# LOAD CLEANED DATA
print("="*60)
print("LOADING CLEANED DATA")
print("="*60)
df = pd.read_csv('../data/preprocessed/city_day_cleaned_v1.csv')
print(f"\n Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

# STEP 1: IDENTIFY NUMERIC COLUMNS
print("\n" + "="*60)
print("STEP 1: IDENTIFY NUMERIC COLUMNS")
print("="*60)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric columns ({len(numeric_cols)}):")
for col in numeric_cols:
    print(f"  • {col}")

# STEP 2: OUTLIER DETECTION (IQR METHOD)
print("\n" + "="*60)
print("STEP 2: OUTLIER DETECTION USING IQR METHOD")
print("="*60)
def detect_outliers_iqr(data, column):
    """Detect outliers using IQR method"""
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return len(outliers), lower_bound, upper_bound
print("\nOutliers detected in each column:")
outlier_summary = {}

for col in numeric_cols:
    if col != 'Month':  # Skip Month (categorical)
        count, lower, upper = detect_outliers_iqr(df, col)
        outlier_summary[col] = {
            'count': count,
            'percentage': (count / len(df)) * 100,
            'lower_bound': lower,
            'upper_bound': upper
        }
        if count > 0:
            print(f"  • {col:15} → {count:6} outliers ({(count/len(df)*100):5.2f}%)")

print("\n Outlier detection complete!")

# STEP 3: HANDLE OUTLIERS - CAP EXTREME VALUES
print("\n" + "="*60)
print("STEP 3: HANDLING OUTLIERS (CAPPING)")
print("="*60)
print("\nApproach: Cap outliers at IQR bounds (don't remove, cap values)")
print("Reason: Pollution data can have extreme spikes (real events)")
df_cleaned = df.copy()
for col in numeric_cols:
    if col != 'Month':
        Q1 = df_cleaned[col].quantile(0.25)
        Q3 = df_cleaned[col].quantile(0.75)
        IQR = Q3 - Q1        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        # Cap values at bounds instead of removing
        df_cleaned[col] = df_cleaned[col].clip(lower=lower_bound, upper=upper_bound)

print(" All outliers capped at IQR bounds!")

# STEP 4: FEATURE SCALING (NORMALIZATION)
print("\n" + "="*60)
print("STEP 4: FEATURE SCALING (0-1 NORMALIZATION)")
print("="*60)
print("\nScaling formula: (value - min) / (max - min)")
print("Result: All features in 0-1 range")

# Separate features and target
features_to_scale = [col for col in numeric_cols if col != 'AQI' and col != 'Month']
target_col = 'AQI'
print(f"\nFeatures to scale ({len(features_to_scale)}):")
for col in features_to_scale:
    print(f"  • {col}")

# Create scaler
scaler = MinMaxScaler(feature_range=(0, 1))

# Scale features
df_cleaned[features_to_scale] = scaler.fit_transform(df_cleaned[features_to_scale])

# Also scale AQI target (optional but good practice)
scaler_aqi = MinMaxScaler(feature_range=(0, 1))
df_cleaned['AQI'] = scaler_aqi.fit_transform(df_cleaned[['AQI']])
print("\n Feature scaling complete!")

# STEP 5: VERIFY SCALING
print("\n" + "="*60)
print("STEP 5: VERIFICATION - SCALED DATA RANGES")
print("="*60)
print("\nScaled data ranges (should all be 0-1):")
print(df_cleaned[features_to_scale + ['AQI']].describe().loc[['min', 'max']].T)

# STEP 6: DATA QUALITY REPORT
print("\n" + "="*60)
print("STEP 6: FINAL DATA QUALITY REPORT")
print("="*60)

print(f"\n Final shape: {df_cleaned.shape}")
print(f" Rows: {len(df_cleaned):,}")
print(f" Columns: {len(df_cleaned.columns)}")
print(f" Missing values: {df_cleaned.isnull().sum().sum()}")
print(f" Duplicates: {df_cleaned.duplicated().sum()}")
print(f" All features scaled: 0-1 range")
print(f" AQI target scaled: 0-1 range")
print("\nData types after scaling:")
print(df_cleaned.dtypes)
print("\nSample of scaled data:")
print(df_cleaned[['City', 'PM2.5', 'PM10', 'NO2', 'Temperature', 'AQI']].head(10))

# STEP 7: SAVE SCALED DATASET
print("\n" + "="*60)
print("STEP 7: SAVING SCALED DATASET")
print("="*60)

df_cleaned.to_csv('../data/preprocessed/city_day_scaled.csv', index=False)
print("\n Saved: data/preprocessed/city_day_scaled.csv")

# Also save the scalers for later use
import pickle
scaler_info = {
    'scaler_features': scaler,
    'scaler_aqi': scaler_aqi,
    'features_scaled': features_to_scale
}

with open('../models/scalers.pkl', 'wb') as f:
    pickle.dump(scaler_info, f)

print(" Saved: models/scalers.pkl (for later predictions)")
print("\n" + "="*60)
print(" DATA PREPROCESSING COMPLETE!")
print("="*60)

LOADING CLEANED DATA

 Loaded: 24,850 rows × 20 columns

STEP 1: IDENTIFY NUMERIC COLUMNS

Numeric columns (17):
  • PM2.5
  • PM10
  • NO
  • NO2
  • NOx
  • NH3
  • CO
  • SO2
  • O3
  • Benzene
  • Toluene
  • Xylene
  • AQI
  • Month
  • Temperature
  • Humidity
  • Wind_Speed

STEP 2: OUTLIER DETECTION USING IQR METHOD

Outliers detected in each column:
  • PM2.5           →   2004 outliers ( 8.06%)
  • PM10            →   1534 outliers ( 6.17%)
  • NO              →   2351 outliers ( 9.46%)
  • NO2             →   1107 outliers ( 4.45%)
  • NOx             →   1937 outliers ( 7.79%)
  • NH3             →   1149 outliers ( 4.62%)
  • CO              →   2303 outliers ( 9.27%)
  • SO2             →   2435 outliers ( 9.80%)
  • O3              →    739 outliers ( 2.97%)
  • Benzene         →   1786 outliers ( 7.19%)
  • Toluene         →   3120 outliers (12.56%)
  • Xylene          →  12337 outliers (49.65%)
  • AQI             →   1358 outliers ( 5.46%)

 Outlier detection complete

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import pickle

# LOAD SCALED DATA
print("="*60)
print("LOADING SCALED DATA FOR TRAIN/TEST SPLIT")
print("="*60)

df = pd.read_csv('../data/preprocessed/city_day_scaled.csv')
print(f"\n Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

# STEP 1: IDENTIFY FEATURES AND TARGET
print("\n" + "="*60)
print("STEP 1: SEPARATE FEATURES AND TARGET")
print("="*60)

# Target column (what we're predicting)
target = 'AQI'

# Feature columns (what we use to predict)
exclude_cols = ['City', 'Date', 'AQI_Bucket', 'AQI']
feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"\nTarget column: {target}")
print(f"Feature columns ({len(feature_cols)}):")
for col in feature_cols:
    print(f"  • {col}")

# Create X (features) and y (target)
X = df[feature_cols]
y = df[target]
print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

# STEP 2: TRAIN/TEST SPLIT (80/20)
print("\n" + "="*60)
print("STEP 2: TRAIN/TEST SPLIT (80/20)")
print("="*60)

# Split: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTraining set:")
print(f"  • X_train shape: {X_train.shape}")
print(f"  • y_train shape: {y_train.shape}")
print(f"  • Rows: {len(X_train):,} (80%)")

print(f"\nTest set:")
print(f"  • X_test shape: {X_test.shape}")
print(f"  • y_test shape: {y_test.shape}")
print(f"  • Rows: {len(X_test):,} (20%)")

# STEP 3: VERIFY TRAIN/TEST SPLIT
print("\n" + "="*60)
print("STEP 3: VERIFICATION")
print("="*60)

print(f"\nTrain set AQI statistics:")
print(f"  • Min: {y_train.min():.4f}")
print(f"  • Max: {y_train.max():.4f}")
print(f"  • Mean: {y_train.mean():.4f}")
print(f"  • Std: {y_train.std():.4f}")

print(f"\nTest set AQI statistics:")
print(f"  • Min: {y_test.min():.4f}")
print(f"  • Max: {y_test.max():.4f}")
print(f"  • Mean: {y_test.mean():.4f}")
print(f"  • Std: {y_test.std():.4f}")

print("\n Train and test sets have similar distributions!")

# STEP 4: SAVE TRAIN/TEST DATASETS
print("\n" + "="*60)
print("STEP 4: SAVING TRAIN/TEST DATASETS")
print("="*60)

# Create DataFrames for saving
train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

# Save as CSV
train_df.to_csv('../data/preprocessed/train_data.csv', index=False)
test_df.to_csv('../data/preprocessed/test_data.csv', index=False)

print("\n Saved train/test CSV files:")
print(f"  • data/preprocessed/train_data.csv ({len(train_df):,} rows)")
print(f"  • data/preprocessed/test_data.csv ({len(test_df):,} rows)")

# STEP 5: SAVE AS NUMPY ARRAYS (FOR ML MODELS)
print("\n" + "="*60)
print("STEP 5: SAVING AS NUMPY ARRAYS (FOR ML MODELS)")
print("="*60)

# Save as numpy arrays (faster for ML models)
train_data = {
    'X_train': X_train.values,
    'y_train': y_train.values,
    'X_test': X_test.values,
    'y_test': y_test.values,
    'feature_names': feature_cols
}

with open('../data/preprocessed/train_test_data.pkl', 'wb') as f:
    pickle.dump(train_data, f)

print("\n Saved: data/preprocessed/train_test_data.pkl")
print(f"  • X_train: {X_train.shape}")
print(f"  • y_train: {y_train.shape}")
print(f"  • X_test: {X_test.shape}")
print(f"  • y_test: {y_test.shape}")

# STEP 6: SUMMARY STATISTICS
print("\n" + "="*60)
print("STEP 6: FINAL SUMMARY")
print("="*60)

print(f"\n DATA PREPROCESSING COMPLETE!")
print(f"\n Original dataset: 29,531 rows")
print(f" After cleaning: {len(df):,} rows")
print(f" Training set: {len(X_train):,} rows (80%)")
print(f" Test set: {len(X_test):,} rows (20%)")
print(f" Features: {len(feature_cols)}")
print(f" Missing values: 0")
print(f" All features scaled: 0-1 range")
print(f" All features ready for ML models")


LOADING SCALED DATA FOR TRAIN/TEST SPLIT

 Loaded: 24,850 rows × 20 columns

STEP 1: SEPARATE FEATURES AND TARGET

Target column: AQI
Feature columns (16):
  • PM2.5
  • PM10
  • NO
  • NO2
  • NOx
  • NH3
  • CO
  • SO2
  • O3
  • Benzene
  • Toluene
  • Xylene
  • Month
  • Temperature
  • Humidity
  • Wind_Speed

X shape: (24850, 16)
y shape: (24850,)

STEP 2: TRAIN/TEST SPLIT (80/20)

Training set:
  • X_train shape: (19880, 16)
  • y_train shape: (19880,)
  • Rows: 19,880 (80%)

Test set:
  • X_test shape: (4970, 16)
  • y_test shape: (4970,)
  • Rows: 4,970 (20%)

STEP 3: VERIFICATION

Train set AQI statistics:
  • Min: 0.0026
  • Max: 1.0000
  • Mean: 0.3748
  • Std: 0.2696

Test set AQI statistics:
  • Min: 0.0000
  • Max: 1.0000
  • Mean: 0.3729
  • Std: 0.2687

 Train and test sets have similar distributions!

STEP 4: SAVING TRAIN/TEST DATASETS

 Saved train/test CSV files:
  • data/preprocessed/train_data.csv (19,880 rows)
  • data/preprocessed/test_data.csv (4,970 rows)

ST